# Create files mapping FACS well and plate to basic allele metadata

This script takes cherry picking and allele info files as input and produces a table relating FACS well and plate IDs to allele IDs and symbols. 

In [ ]:
import pandas as pd

In [ ]:
dualipa_inputs = "../1_inputs"
allele_collection_inputs = "../../1_allele_collection/1_inputs"
dualipa_outputs = "../3_outputs"

In [ ]:
# REPLACE THIS WITH TONG'S FILE ONCE HE ADD MUTATION ID
allele_info_df = pd.read_csv(f"{inputs_dir}/VUS_wt_allele_collection_entry_plates_allele_infos.csv", index_col=0)

Vidal lab ran experiment once and analyzed the data. Some alleles didn't pass the qa/qc thresholds or got no data. Batch5 is is a re-run for all alleles that failed in the first set of batches. Here, we process everything together from scratch. An allele should always be compared to the wt on the same batch and plate. The summary stats can then be compared across batches. 

In [ ]:
# Read in cherry picking file for first set of batches
# GET CHERRY PICKING FILE

# Cherry picking file for second round of batches
pdest_layout_b5_df = pd.read_csv(f'{inputs_dir}/summary_CP_VUSDUALXP_batch5.csv', index_col=0)
col = ['orf_id', 'mut_id', 'node_type','dest_pla_id', 'dest_pla', 'dest_well']
pdest_layout_b5_df = pdest_layout_b5_df[col]
pdest_layout_b5_df.rename(columns = {'dest_pla':'exp_plate', 'dest_well':'exp_well'}, inplace=True)

pdest_layout_df = pd.concat([pdest_layout_b5_df, pdest_layout_df])

In [ ]:
# Use mutation and orf ID to add nt change, aa change, and clinvar label
# REDO WITH TONG'S UPDATED ALLELE INFO FILE
pdest_layout_df['nt_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][0] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]
pdest_layout_df['aa_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][1] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]

# REDO WITH CHLOE'S UPDATED CLINVAR ANNOTATIONS
pdest_layout_df['clinical_significance'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][2] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]

In [ ]:
pdest_layout_df.to_csv(f'{dualipa_outputs}/pdest_experimental_layout.csv', index=None)